In [1]:
using Revise
using JET
using DataFrames
using Pkg
Pkg.activate("..")

using PowerFlowData
using StructArrays
using InlineStrings

using Powerful
using Powerful.Models

using Powerful.PowerCore

  Activating project at `~/repos/Powerful.jl`


In [2]:
case = PowerFlowData.parse_network(joinpath(@__DIR__, "..", "cases", "ieee14.raw"));

In [3]:
ALL_MODELS = MODEL_REGISTRY[].models

models = load_system(case, ALL_MODELS);

In [4]:
am = AddressManager();

In [5]:
sys = SystemModel(am, models);

In [6]:
for m in ALL_MODELS
    sym = Symbol(m)
    allocate_model!(
        am,
        sym, 
        model_metadata(m), 
        length(sys.models[sym]),
    )
end

In [7]:
for m in ALL_MODELS
    retrieve_addresses(sys.address_manager, model_metadata(m), sys.models)
end

In [8]:
am.addresses

Dict{Tuple{AddressableType, Symbol, Symbol}, Vector{UInt64}} with 4 entries:
  (AlgebRes(), :Bus, :q)     => [0x000000000000000f, 0x0000000000000010, 0x0000…
  (AlgebVar(), :Bus, :v)     => [0x000000000000000f, 0x0000000000000010, 0x0000…
  (AlgebRes(), :Bus, :p)     => [0x0000000000000001, 0x0000000000000002, 0x0000…
  (AlgebVar(), :Bus, :theta) => [0x0000000000000001, 0x0000000000000002, 0x0000…

In [9]:
am.retrieved[:Transformer2W].by_type

Dict{AddressableType, Dict{Symbol, Vector{UInt64}}} with 5 entries:
  StateVar()    => Dict()
  AlgebVar()    => Dict(:theta_j=>[0x0000000000000005, 0x0000000000000007, 0x00…
  ObservedVar() => Dict()
  StateRes()    => Dict()
  AlgebRes()    => Dict(:q_j=>[0x0000000000000013, 0x0000000000000015, 0x000000…

In [10]:
am.retrieved[:PV].by_type

Dict{AddressableType, Dict{Symbol, Vector{UInt64}}} with 5 entries:
  StateVar()    => Dict()
  AlgebVar()    => Dict(:v=>[0x000000000000000f, 0x0000000000000010, 0x00000000…
  ObservedVar() => Dict()
  StateRes()    => Dict()
  AlgebRes()    => Dict(:p=>[0x0000000000000001, 0x0000000000000002, 0x00000000…

In [11]:
am.retrieved[:Branch].by_type[AlgebRes()]

Dict{Symbol, Vector{UInt64}} with 4 entries:
  :q_j => [0x0000000000000010, 0x0000000000000011, 0x0000000000000012, 0x000000…
  :p_j => [0x0000000000000002, 0x0000000000000003, 0x0000000000000004, 0x000000…
  :q_i => [0x000000000000000f, 0x0000000000000010, 0x0000000000000011, 0x000000…
  :p_i => [0x0000000000000001, 0x0000000000000002, 0x0000000000000003, 0x000000…

In [12]:
am.retrieved[:PQ].by_type[AlgebRes()]

Dict{Symbol, Vector{UInt64}} with 2 entries:
  :p => [0x0000000000000002, 0x0000000000000003, 0x0000000000000004, 0x00000000…
  :q => [0x0000000000000010, 0x0000000000000011, 0x0000000000000012, 0x00000000…

In [13]:
am.retrieved[:PQ].by_type[AlgebVar()]

Dict{Symbol, Vector{UInt64}} with 2 entries:
  :v     => [0x0000000000000010, 0x0000000000000011, 0x0000000000000012, 0x0000…
  :theta => [0x0000000000000002, 0x0000000000000003, 0x0000000000000004, 0x0000…

In [14]:
am.allocations[:Bus].by_type

Dict{AddressableType, Dict{Symbol, Vector{UInt64}}} with 5 entries:
  StateVar()    => Dict()
  AlgebVar()    => Dict(:v=>[0x000000000000000f, 0x0000000000000010, 0x00000000…
  ObservedVar() => Dict()
  StateRes()    => Dict()
  AlgebRes()    => Dict(:p=>[0x0000000000000001, 0x0000000000000002, 0x00000000…

In [15]:
am.allocations[:Bus].by_type

Dict{AddressableType, Dict{Symbol, Vector{UInt64}}} with 5 entries:
  StateVar()    => Dict()
  AlgebVar()    => Dict(:v=>[0x000000000000000f, 0x0000000000000010, 0x00000000…
  ObservedVar() => Dict()
  StateRes()    => Dict()
  AlgebRes()    => Dict(:p=>[0x0000000000000001, 0x0000000000000002, 0x00000000…

In [16]:
case

Network (v33) with 18 data categories:
 CaseID: (ic = 0, sbase = 100.0, rev = 33, xfrrat = 0, nxfrat = 1, basfrq = 60.0)
 Buses with 14 records
 Loads with 11 records
 FixedShunts with 0 records
 Generators with 5 records
 Branches with 16 records
 Transformers with 4 records
 AreaInterchanges with 2 records
 TwoTerminalDCLines with 0 records
 VSCDCLines with 0 records
 SwitchedShunts with 2 records
 ImpedanceCorrections with 0 records
 MultiTerminalDCLines with 0 records
 MultiSectionLineGroups with 0 records
 Zones with 2 records
 InterAreaTransfers with 0 records
 Owners with 2 records
 FACTSDevices with 0 records

In [17]:
describe(DataFrame(case.branches),:eltype)

Row,variable,eltype
,Symbol,Type
1,i,Int32
2,j,Int32
3,ckt,String3
4,r,Float64
5,x,Float64
6,b,Float64
7,rate_a,Float64
8,rate_b,Float64
9,rate_c,Float64


In [18]:
using DataFrames

DataFrame(case.owners)

Row,i,owname
,Int16,String15
1,1,OWNER1
2,2,OWNER2


In [19]:
DataFrame(case.zones)

Row,i,zoname
,Int16,String15
1,1,ZONE1
2,2,ZONE2


In [20]:
DataFrame(case.area_interchanges)

Row,i,isw,pdes,ptol,arname
,Int16,Int32,Float64,Float64,String15
1,1,1,0.0,999.99,AREA1
2,2,0,0.0,10.0,AREA2


In [21]:
generators = DataFrame(case.generators)

Row,i,id,pg,qg,qt,qb,vs,ireg,mbase,zr,zx,rt,xt,gtap,stat,rmpct,pt,pb,o1,f1,o2,f2,o3,f3,o4,f4,wmod,wpf
,Int32,String3,Float64,Float64,Float64,Float64,Float64,Int32,Float64,Float64,Float64,Float64,Float64,Float64,Bool,Float64,Float64,Float64,Int16,Float64?,Int16?,Float64?,Int16?,Float64?,Int16?,Float64?,Int8?,Float64?
1,1,1,81.442,1.962,100.0,-50.0,1.03,0,100.0,0.0,0.23,0.0,0.0,1.0,true,100.0,200.0,50.0,1,1.0,missing,missing,missing,missing,missing,missing,missing,missing
2,2,1,40.0,15.0,15.0,-40.0,1.03,0,100.0,0.0,0.13,0.0,0.0,1.0,true,100.0,50.0,10.0,1,1.0,missing,missing,missing,missing,missing,missing,missing,missing
3,3,1,40.0,15.0,15.0,-10.0,1.01,0,100.0,0.0,0.13,0.0,0.0,1.0,true,100.0,50.0,10.0,1,1.0,missing,missing,missing,missing,missing,missing,missing,missing
4,6,1,30.0,10.0,10.0,-6.0,1.03,0,100.0,0.0,0.12,0.0,0.0,1.0,true,100.0,50.0,10.0,1,1.0,missing,missing,missing,missing,missing,missing,missing,missing
5,8,1,35.0,10.0,10.0,-6.0,1.03,0,100.0,0.0,0.12,0.0,0.0,1.0,true,100.0,50.0,10.0,1,1.0,missing,missing,missing,missing,missing,missing,missing,missing


In [22]:
print(describe(generators, :eltype))

28×2 DataFrame
 Row │ variable  eltype                  
     │ Symbol    Type                    
─────┼───────────────────────────────────
   1 │ i         Int32
   2 │ id        String3
   3 │ pg        Float64
   4 │ qg        Float64
   5 │ qt        Float64
   6 │ qb        Float64
   7 │ vs        Float64
   8 │ ireg      Int32
   9 │ mbase     Float64
  10 │ zr        Float64
  11 │ zx        Float64
  12 │ rt        Float64
  13 │ xt        Float64
  14 │ gtap      Float64
  15 │ stat      Bool
  16 │ rmpct     Float64
  17 │ pt        Float64
  18 │ pb        Float64
  19 │ o1        Int16
  20 │ f1        Union{Missing, Float64}
  21 │ o2        Union{Missing, Int16}
  22 │ f2        Union{Missing, Float64}
  23 │ o3        Union{Missing, Int16}
  24 │ f3        Union{Missing, Float64}
  25 │ o4        Union{Missing, Int16}
  26 │ f4        Union{Missing, Float64}
  27 │ wmod      Union{Missing, Int8}
  28 │ wpf       Union{Missing, Float64}

In [24]:
?Generators

search: Generators generators Base.Generator Iterators numerator Pkg.generate



```julia
struct Generators <: PowerFlowData.Records
```

Each network bus to be represented as a generator or plant bus in PSS/E must be specified in a generator data record. In particular, each bus specified in the bus data input with a type code of two (2) or three (3) must have a generator data record entered for it.

# Fields

  * `i::Vector{Int32}`: Bus number, or extended bus name enclosed in single quotes.
  * `id::Vector{String3}`: One- or two-character uppercase non blank alphanumeric machine identifier used to distinguish among multiple machines at bus "I". It is recommended that, at buses for which a single machine is present, the machine be designated as having the machine identifier ’1’. ID = ’1’ by default.

  * `pg::Vector{Float64}`: Generator active power output; entered in MW. PG = 0.0 by default.
  * `qg::Vector{Float64}`: Generator reactive power output; entered in Mvar. QG needs to be entered only if the case, as read in, is to be treated as a solved case. QG = 0.0 by default.

  * `qt::Vector{Float64}`: Maximum generator reactive power output; entered in Mvar. For fixed output gen- erators (i.e., nonregulating), QT must be equal to the fixed Mvar output. QT = 9999.0 by default.

  * `qb::Vector{Float64}`: Minimum generator reactive power output; entered in Mvar. For fixed output generators, QB must be equal to the fixed Mvar output. QB = -9999.0 by default.

  * `vs::Vector{Float64}`: Regulated voltage setpoint; entered in pu. VS = 1.0 by default.
  * `ireg::Vector{Int32}`: Bus number, or extended bus name enclosed in single quotes, of a remote type 1 or 2 bus whose voltage is to be regulated by this plant to the value specified by VS. If bus IREG is other than a type 1 or 2 bus, bus "I" regulates its own voltage to the value specified by VS. IREG is entered as zero if the plant is to regulate its own voltage and must be zero for a type three (swing) bus. IREG = 0 by default.

  * `mbase::Vector{Float64}`: Total MVA base of the units represented by this machine; entered in MVA. This quantity is not needed in normal power flow and equivalent onstruction work, but is required for switching studies, fault analysis, and dynamic simulation. MBASE = system base MVA by default.

  * `zr::Vector{Float64}`: Complex machine impedance, ZSORCE; entered in pu on MBASE base. This data is not needed in normal power flow and equivalent construction work, but is required for switching studies, fault analysis, and dynamic simulation. For dynamic simulation, this impedance must be set equal to the unsaturated subtransient impedance for those generators to be modeled by subtransient level machine models, and to unsaturated transient impedance for those to be modeled by classical or transient level models. For short-circuit studies, the saturated subtransient or transient impedance should be used. ZR = 0.0 by default.

  * `zx::Vector{Float64}`: See `zr`. ZX = 1.0 by default.
  * `rt::Vector{Float64}`: Step-up transformer impedance, XTRAN; entered in pu on MBASE base. XTRAN should be entered as zero if the step-up transformer is explicitly modeled as a network branch and bus "I" is the terminal bus. RT+jXT = 0.0 by default.

  * `xt::Vector{Float64}`: See `rt`. RT+jXT = 0.0 by default.
  * `gtap::Vector{Float64}`: Step-up transformer off-nominal turns ratio; entered in pu. GTAP is used only if XTRAN is nonzero. GTAP = 1.0 by default.

  * `stat::Vector{Bool}`: Initial machine status of one for in-service and zero for out-of-service. STAT = 1 by default.

  * `rmpct::Vector{Float64}`: Percent of the total Mvar required to hold the voltage at the bus controlled by this bus "I" that are to be contributed by the generation at bus "I"; RMPCT must be positive. RMPCT is needed if IREG specifies a valid remote bus and there is more than one local or remote voltage controlling device (plant, switched shunt, FACTS device shunt element, or VSC DC line converter) controlling the voltage at bus IREG to a setpoint. RMPCT is needed also if bus "I" itself is being controlled locally or remotely by one or more other setpoint mode voltage controlling devices. RMPCT = 100.0 by default.

  * `pt::Vector{Float64}`: Maximum generator active power output; entered in MW. PT = 9999.0 by default.
  * `pb::Vector{Float64}`: Minimum generator active power output; entered in MW. PB = -9999.0 by default.
  * `o1::Vector{Int16}`: Owner number (1 through the maximum number of owners at the current size level). Each machine may have up to four owners. See [`Owners`](@ref). By default, `o1` is the owner to which bus `i` is assigned and `o2`, `o3`, and `o4` are zero.

  * `f1::Vector{Union{Missing, Float64}}`: Fraction of total ownership assigned to owner `oi`; each `fi` must be positive. The `fi` values are normalized such that they sum to 1.0 before they are placed in the working case. By default, each `fi` is 1.0.

  * `o2::Vector{Union{Missing, Int16}}`
  * `f2::Vector{Union{Missing, Float64}}`
  * `o3::Vector{Union{Missing, Int16}}`
  * `f3::Vector{Union{Missing, Float64}}`
  * `o4::Vector{Union{Missing, Int16}}`
  * `f4::Vector{Union{Missing, Float64}}`
  * `wmod::Vector{Union{Missing, Int8}}`: Wind machine control mode; `wmod` is used to indicate whether a machine is a wind machine, and, if it is, the type of reactive power limits to be imposed.

      * 0 for a machine that is not a wind machine.
      * 1 for a wind machine for which reactive power limits are specified by QT and QB.
      * 2 for a wind machine for which reactive power limits are determined from the machine’s active power output and `wpf`; limits are of equal magnitude and opposite sign.
      * 3 for a wind machine with a fixed reactive power setting determined from the machine’s active power output and `wpf`; when `wpf` is positive, the machine’s reactive power has the same sign as its active power; when `wpf` is negative, the machine’s reactive power has the opposite sign of its active power.

    `wmod` = 0 by default.

  * `wpf::Vector{Union{Missing, Float64}}`: Power factor used in calculating reactive power limits or output when `wmod` is 2 or 3. `wpf` = 1.0 by default.


In [25]:
?Bus

search: Bus Buses Bounds BusNum BusVec Base



```julia
mutable struct Bus{Tv} <: Powerful.Models.AbstractBus{Tv}
```

Network bus data records.

# Fields

  * `i::Int32`: Bus number (1 to 999997).
  * `name::String15`: Alphanumeric identifier assigned to bus 'I'.

    The name may be up to twelve characters and must be enclosed in single quotes. NAME may contain any combination of blanks, uppercase letters, numbers and special characters, but the first character must not be a minus sign.

  * `basekv::Any`: Bus base voltage in kV.
  * `ide::Int8`: Bus type code:

      * 1 - load bus or other bus without any generator boundary condition.
      * 2 - generator or plant bus either regulating voltage or with a fixed reactive power (Mvar). A generator that reaches its reactive power limit will no longer control voltage but rather hold reactive power at its limit.
      * 3 - swing bus or slack bus. It has no power or reactive limits and regulates voltage at a fixed reference angle.
      * 4 - disconnected or isolated bus.

  * `area::Int16`: Area number. 1 through the maximum number of areas at the current size level.
  * `zone::Int16`: Zone number. 1 through the maximum number of zones at the current size level.
  * `owner::Int16`: Owner number. 1 through the maximum number of owners at the current size level.
  * `vm::Any`: Bus voltage magnitude in pu.
  * `va::Any`: Bus voltage angle in degrees.
  * `nvhi::Any`: Normal voltage magnitude high limit in pu.
  * `nvlo::Any`: Normal voltage magnitude low limit in pu.
  * `evhi::Any`: Emergency voltage magnitude high limit in pu.
  * `evlo::Any`: Emergency voltage magnitude low limit in pu.


In [28]:
?Powerful.Models.AreaInterchange

!!! warning
    The following bindings may be internal; they may change or be removed in future versions:

      * `Powerful.Models`


```julia
struct AreaInterchange{Tv} <: AbstractModel
```

Area interchange data records.

# Fields

  * `i::Int16`: Area number (1 through the maximum number of areas at the current size level)

  * `isw::Int32`: Area slack bus number for interchange control, given by its bus number or extended bus name enclosed in single quotes. The bus must be a generator (type two) bus in the specified area. Any area containing a system swing bus (type three) must have either that swing bus or a bus number of zero specified for its area slack bus number. `isw` = 0 by default.

  * `pdes::Any`: Desired net interchange leaving the area (export) in MW.

  * `ptol::Any`: Interchange tolerance bandwidth; entered in MW. `ptol` = 10.0 by default.

  * `arname::String15`: Alphanumeric identifier assigned to area I. The name may contain up to twelve characters. `arname` is set to twelve blanks by default.


In [29]:
length(Base.Docs.doc(BusVec))

7

In [30]:
Base.doc

doc (generic function with 6 methods)

In [31]:
string(Docs.getdoc(Bus))

"nothing"

In [32]:
?Bus

search: Bus Buses Bounds BusNum BusVec Base



```julia
mutable struct Bus{Tv} <: Powerful.Models.AbstractBus{Tv}
```

Network bus data records.

# Fields

  * `i::Int32`: Bus number (1 to 999997).
  * `name::String15`: Alphanumeric identifier assigned to bus 'I'.

    The name may be up to twelve characters and must be enclosed in single quotes. NAME may contain any combination of blanks, uppercase letters, numbers and special characters, but the first character must not be a minus sign.

  * `basekv::Any`: Bus base voltage in kV.
  * `ide::Int8`: Bus type code:

      * 1 - load bus or other bus without any generator boundary condition.
      * 2 - generator or plant bus either regulating voltage or with a fixed reactive power (Mvar). A generator that reaches its reactive power limit will no longer control voltage but rather hold reactive power at its limit.
      * 3 - swing bus or slack bus. It has no power or reactive limits and regulates voltage at a fixed reference angle.
      * 4 - disconnected or isolated bus.

  * `area::Int16`: Area number. 1 through the maximum number of areas at the current size level.
  * `zone::Int16`: Zone number. 1 through the maximum number of zones at the current size level.
  * `owner::Int16`: Owner number. 1 through the maximum number of owners at the current size level.
  * `vm::Any`: Bus voltage magnitude in pu.
  * `va::Any`: Bus voltage angle in degrees.
  * `nvhi::Any`: Normal voltage magnitude high limit in pu.
  * `nvlo::Any`: Normal voltage magnitude low limit in pu.
  * `evhi::Any`: Emergency voltage magnitude high limit in pu.
  * `evlo::Any`: Emergency voltage magnitude low limit in pu.


In [33]:
case

Network (v33) with 18 data categories:
 CaseID: (ic = 0, sbase = 100.0, rev = 33, xfrrat = 0, nxfrat = 1, basfrq = 60.0)
 Buses with 14 records
 Loads with 11 records
 FixedShunts with 0 records
 Generators with 5 records
 Branches with 16 records
 Transformers with 4 records
 AreaInterchanges with 2 records
 TwoTerminalDCLines with 0 records
 VSCDCLines with 0 records
 SwitchedShunts with 2 records
 ImpedanceCorrections with 0 records
 MultiTerminalDCLines with 0 records
 MultiSectionLineGroups with 0 records
 Zones with 2 records
 InterAreaTransfers with 0 records
 Owners with 2 records
 FACTSDevices with 0 records

In [34]:
print(describe(DataFrame(case.transformers), :eltype))

43×2 DataFrame
 Row │ variable  eltype                   
     │ Symbol    Type                     
─────┼────────────────────────────────────
   1 │ i         Int32
   2 │ j         Int32
   3 │ k         Int32
   4 │ ckt       String3
   5 │ cw        Int8
   6 │ cz        Int8
   7 │ cm        Int8
   8 │ mag1      Float64
   9 │ mag2      Float64
  10 │ nmetr     Int8
  11 │ name      String15
  12 │ stat      Bool
  13 │ o1        Int16
  14 │ f1        Float64
  15 │ o2        Union{Missing, Int16}
  16 │ f2        Union{Missing, Float64}
  17 │ o3        Union{Missing, Int16}
  18 │ f3        Union{Missing, Float64}
  19 │ o4        Union{Missing, Int16}
  20 │ f4        Union{Missing, Float64}
  21 │ vecgrp    Union{Missing, String15}
  22 │ r1_2      Float64
  23 │ x1_2      Float64
  24 │ sbase1_2  Float64
  25 │ windv1    Float64
  26 │ nomv1     Float64
  27 │ ang1      Float64
  28 │ rata1     Float64
  29 │ ratb1     Float64
  30 │ ratc1     Float64
  31 │ cod1      Int8

In [35]:
case.transformers

Transformers with 4 records, 43 columns:
────────────────────────────────────────────────────────────────────────────────
 i  j  k  ckt  cw  cz  cm  mag1  mag2  nmetr  name  stat  o1   f1  o2   f2  o3 ⋯
────────────────────────────────────────────────────────────────────────────────
 4  7  0    1   1   1   1   0.0   0.0      2        true   1  1.0   0  1.0   0 ⋯
 4  9  0    1   1   1   1   0.0   0.0      2        true   1  1.0   0  1.0   0 ⋯
 6  5  0    1   1   1   1   0.0   0.0      1        true   1  1.0   0  1.0   0 ⋯
 8  7  0    1   1   1   1   0.0   0.0      1        true   1  1.0   0  1.0   0 ⋯
────────────────────────────────────────────────────────────────────────────────
                                                              26 columns omitted

In [36]:
?Transformers

search: Transformers Transformer2W Transformer2WNum Transformer2WVec transform



```julia
struct Transformers <: PowerFlowData.Records
```

Each AC transformer to be represented in PSS/E is introduced through transformer data records that specify all the data required to model transformers in power flow calculations, with one exception.

That exception is a set of ancillary data, comprising transformer impedance correction records, which define the manner in which transformer impedance changes as off-nominal turns ratio or phase shift angle is adjusted. Those data records are described in Transformer Impedance Correction Records, [`ImpedanceCorrections`](@ref).

Both two-winding and three-winding transformers are specified in the transformer data records. The data records for the two-winding transformer are common to the three-winding transformer; the data block for two-winding transformers is a subset of the data required for three-winding transformers.

# Fields

  * `i::Vector{Int32}`: The bus number, or extended bus name enclosed in single quotes, of the bus to which the first winding is connected. The transformer’s magnetizing admittance is modeled on winding one. The first winding is the only winding of a two-winding transformer whose tap ratio or phase shift angle may be adjusted by the power flow solution activities; any winding(s) of a three-winding transformer may be adjusted. No default is allowed.

  * `j::Vector{Int32}`: The bus number, or extended bus name enclosed in single quotes, of the bus to which the second winding is connected. This winding may have a fixed, off-nominal tap ratio assigned to it. No default is allowed.

  * `k::Vector{Int32}`: The bus number, or extended bus name enclosed in single quotes, of the bus to which the third winding is connected. Zero is used to indicate that no third winding is present. *Always equal to zero for a two-winding transformer.*

  * `ckt::Vector{String3}`: One- or two-character uppercase nonblank alphanumeric transformer circuit identifier; the first character of `ckt` must not be an ampersand ('&').

  * `cw::Vector{Int8}`: The winding data I/O code which defines the units in which the turns ratios `windv1` and `windv2` are specified (the units of `rma1` and `rmi1` are also governed by `cw` when `|cod1|` is 1 or 2):

      * 1 for off-nominal turns ratio in pu of winding bus base voltage;
      * 2 for winding voltage in kV.

    `cw` = 1 by default.

  * `cz::Vector{Int8}`: The impedance data I/O code that defines the units in which the winding impedances `r1_2` and `x1_2` are specified:

      * 1 for resistance and reactance in pu on system base quantities;
      * 2 for resistance and reactance in pu on a specified base MVA and winding bus base voltage;
      * 3 for transformer load loss in watts and impedance magnitude in pu on a specified base MVA and winding bus base voltage.

    `cz` = 1 by default.

  * `cm::Vector{Int8}`: The magnetizing admittance I/O code that defines the units in which `mag1` and `mag2` are specified:

      * 1 for complex admittance in pu on system base quantities;
      * 2 for no load loss in watts and exciting current in pu on winding one to two base MVA and nominal voltage.

    `cm` = 1 by default.

  * `mag1::Vector{Float64}`: When `cm` is 1, `mag1` is the magnetizing conductance in pu on system base quantities; when `cm` is 2, `mag1` is the no load loss in watts. `mag1` = 0.0 by default.

  * `mag2::Vector{Float64}`: When `cm` is 1, `mag2` is the magnetizing susceptance in pu on system base quantities; when `cm` is 2, `mag2` is the exciting current in pu on winding one to two base MVA (`sbase1_2`) and nominal voltage (`nomv1`). `mag2` = 0.0 by default.

  * `nmetr::Vector{Int8}`: The nonmetered end code of either:

      * 1 (for the winding one bus), or
      * 2 (for the winding two bus).

    `nmetr` = 2 by default.

  * `name::Vector{String15}`: An alphanumeric identifier assigned to the transformer. The name may be up to twelve characters. `name` may contain any combination of blanks, uppercase letters, numbers and special characters. `name` is twelve blanks by default.

  * `stat::Vector{Bool}`: The initial transformer status, where 1 designates in-service and 0 designates out-of-service. `stat` = 1 by default.

  * `o1::Vector{Int16}`: An owner number; (1 through the maximum number of owners at the current size level). Each transformer may have up to four owners. See [`Owners`](@ref). By default, O1 is the owner to which bus "I" is assigned

  * `f1::Vector{Float64}`: The fraction of total ownership assigned to owner `Oi`; each Fi must be positive. The Fi values are normalized such that they sum to 1.0 before they are placed in the working case. By default, each `fi` is 1.0.

  * `o2::Vector{Union{Missing, Int16}}`
  * `f2::Vector{Union{Missing, Float64}}`
  * `o3::Vector{Union{Missing, Int16}}`
  * `f3::Vector{Union{Missing, Float64}}`
  * `o4::Vector{Union{Missing, Int16}}`
  * `f4::Vector{Union{Missing, Float64}}`
  * `vecgrp::Vector{Union{Missing, String15}}`: Alphanumeric identifier specifying vector group based on transformer winding connections and phase angles. `vecgrp` value is used for information purpose only. `vecgrp` is 12 blanks by default.

  * `r1_2::Vector{Float64}`: The measured impedance of the transformer between the buses to which its first and second windings are connected (see also `x1_2`).

      * When `cz` is 1, `r1_2` is the resistance in pu on system base quantities;
      * when `cz` is 2, `r1_2` is the resistance in pu on winding one to two base MVA (`sbase1_2`) and winding one bus base voltage;
      * when `cz` is 3, `r1_2` is the load loss in watts.

    `r1_2` = 0.0  by default.

  * `x1_2::Vector{Float64}`: The measured impedance of the transformer between the buses to which its first and second windings are connected (see also `r1_2`).

      * When `cz` is 1, `x1_2` is the reactance in pu on system base quantities;
      * when `cz` is 2, `x1_2` is the reactance in pu on winding one to two base MVA (`sbase1_2`) and winding one bus base voltage;
      * when `cz` is 3, `x1_2` is the impedance magnitude in pu on winding one to two base MVA (`sbase1_2`) and winding one bus base voltage.

    `x1_2` has no default.

  * `sbase1_2::Vector{Float64}`: The winding one to two base MVA of the transformer. `sbase1_2` = `sbase` (the system base MVA) by default.

  * `r2_3::Vector{Union{Missing, Float64}}`: The measured impedance of a three-winding transformer between the buses to which its second and third windings are connected (see also `x2_3`).

      * When `cz` is 1, `r2_3` is the resistance in pu on system base quantities;
      * when `cz` is 2, `r2_3` is the resistance in pu on winding two to three base MVA (`sbase2_3`) and winding two bus base voltage;
      * when `cz` is 3, `r2_3` is the load loss in watts

    `r2_3` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `x2_3::Vector{Union{Missing, Float64}}`: The measured impedance of a three-winding transformer between the buses to which its second and third windings are connected (see also `x2_3`).

      * When `cz` is 1, `x2_3` is the reactance in pu on system base quantities;
      * when `cz` is 2, `x2_3` is the reactance in pu on winding one to two base MVA (`sbas2_3`) and winding one bus base voltage;
      * when `cz` is 3, `x2_3` is the impedance magnitude in pu on winding two to three base MVA (`sbase2_3`) and winding two bus base voltage.

    `x2_3` has no default. *Ignored for a two-winding transformer.*

  * `sbase2_3::Vector{Union{Missing, Float64}}`: The winding two to three base MVA of a three-winding transformer; ignored for a two-winding transformer. `sbase2_3` = `sbase` (the system base MVA) by default. *Ignored for a two-winding transformer.*

  * `r3_1::Vector{Union{Missing, Float64}}`: The measured impedance of a three-winding transformer between the buses to which its third and first windings are connected (see also `x3_1`).

      * When `cz` is 1, `r3_1` is the resistance in pu on system base quantities;
      * when `cz` is 2, `r3_1` is the resistance in pu on winding three to one base MVA (`sbase3_1`) and winding three bus base voltage;
      * when `cz` is 3, `r3_1` is the load loss in watts

    `r3_1` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `x3_1::Vector{Union{Missing, Float64}}`: The measured impedance of a three-winding transformer between the buses to which its third and first windings are connected (see also `x3_1`).

      * When `cz` is 1, `x3_1` is the reactance in pu on system base quantities;
      * when `cz` is 2, `x3_1` is the reactance in pu on winding three to one base MVA (`sbas3_1`) and winding three bus base voltage;
      * when `cz` is 3, `x3_1` is the impedance magnitude in pu on winding three to one base MVA (`sbase3_1`) and winding three bus base voltage.

    `x3_1` has no default. *Ignored for a two-winding transformer.*

  * `sbase3_1::Vector{Union{Missing, Float64}}`: The winding three to one base MVA of a three-winding transformer. `sbase3_1` = `sbase` (the system base MVA) by default. *Ignored for a two-winding transformer.*

  * `vmstar::Vector{Union{Missing, Float64}}`: The voltage magnitude at the hidden star point bus; entered in pu. `vmstar` = 1.0 by default. *Ignored for a two-winding transformer.*

  * `anstar::Vector{Union{Missing, Float64}}`: The bus voltage phase angle at the hidden star point bus; entered in degrees. `anstar` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `windv1::Vector{Float64}`: When `cw` is 1, `windv1` is the winding one off-nominal turns ratio in pu of winding one bus base voltage, and windv1 = 1.0 by default. When `cw` is 2, `windv1` is the actual winding one voltage in kV, and `windv1` is equal to the base voltage of bus "I" by default.

  * `nomv1::Vector{Float64}`: The nominal (rated) winding one voltage in kV, or zero to indicate that nominal winding one voltage is to be taken as the base voltage of bus "I". `nomv1` is used only in converting magnetizing data between per unit admittance values and physical units when `cm` is 2. `nomv1` = 0.0 by default.

  * `ang1::Vector{Float64}`: The winding one phase shift angle in degrees. `ang1` is positive for a positive phase shift from the winding one side to the winding two side (for a two-winding transformer). `ang1` must be greater than -180.0 and less than or equal to +180.0. `ang1` = 0.0 by default.

  * `rata1::Vector{Float64}`: The first winding’s first rating entered in MVA (not current expressed in MVA).

  * `ratb1::Vector{Float64}`: The first winding’s second rating entered in MVA (not current expressed in MVA).

  * `ratc1::Vector{Float64}`: The first winding’s third rating entered in MVA (not current expressed in MVA).

  * `cod1::Vector{Int8}`: The transformer control mode for automatic adjustments of the winding one tap or phase shift angle during power flow solutions:

      * 0 for no control (fixed tap and phase shift);
      * ±1 for voltage control;
      * ±2 for reactive power flow control;
      * ±3 for active power flow control;
      * ±4 for control of a DC line quantity.

    If the control mode is entered as a positive number, automatic adjustment of this transformer winding is enabled when the corresponding adjustment is activated during power flow solutions; a negative control mode suppresses the automatic adjustment of this transformer winding. `cod1` = 0 by default.

  * `cont1::Vector{Int32}`: The bus number, or extended bus name enclosed in single quotes, of the bus whose voltage is to be controlled by the transformer turns ratio adjustment option of the power flow solution activities when `cod1` is 1.

    `cont1` should be non-zero only for voltage controlling transformer windings. `cont1` may specify a bus other than "I", "J", or "K"; in this case, the sign of `cont1` defines the location of the controlled bus relative to the transformer winding.

    If `cont1` is entered as a positive number, the ratio is adjusted as if bus `cont1` is on the winding two side of the transformer; if `cont1` is entered as a negative number, the ratio is adjusted as if bus `|cont1|` is on the winding one side of the transformer. `cont1` = 0 by default.

  * `rma1::Vector{Float64}`: `rma1` is the upper limit (and `rmi1` the lower limit) of either:

      * Off-nominal turns ratio in pu of winding one bus base voltage when `|cod1|` is 1 or 2 and `cw` is 1; `rma1` = 1.1 and `rmi1` = 0.9 by default.
      * Actual winding one voltage in kV when `|cod1|` is 1 or 2 and `cw` is 2. No default is allowed.
      * Phase shift angl e in degrees when `|cod1|` is 3. No default is allowed.
      * Not used when `|cod1|` is 0 or 4;

    `rma1` = 1.1 and `rmi1` = 0.9 by default.

  * `rmi1::Vector{Float64}`: The lower limit to `rma1`'s upper limit. See `rma1` for details.
  * `vma1::Vector{Float64}`: `vma1` is the upper limit (and `vmi1` the lower limit) of either:

      * Voltage at the controlled bus (bus `|cont1|`) in pu when `|cod1|` is 1. `vma1` = 1.1 and `vmi1` = 0.9 by default.
      * Reactive power flow into the transformer at the winding one bus end in Mvar when `|cod1|` is 2. no default is allowed.
      * Active power flow into the transformer at the winding one bus end in MW when `|cod1|` is 3. no default is allowed.
      * Not used when `|cod1|` is 0 or 4; `vma1` = 1.1 and `vmi1` = 0.9 by default.

  * `vmi1::Vector{Float64}`: The lower limit to `vma1`'s upper limit. See `vma1` for details.
  * `ntp1::Vector{Int16}`: The number of tap positions available; used when `cod1` is 1 or 2. `ntp1` must be between 2 and 9999. `ntp1` = 33 by default.

  * `tab1::Vector{Int64}`: The number of a transformer impedance correction record if this transformer winding’s impedance is to be a function of either off-nominal turns ratio or phase shift angle, or 0 if no transformer impedance correction is to be applied to this transformer winding. See [`ImpedanceCorrections`](@ref). `tab1` = 0 by default.

  * `cr1::Vector{Float64}`: The load drop compensation impedance for voltage controlling transformers entered in pu on system base quantities; used when `cod1` is 1. `cr1` + j`cx1` = 0.0 by default.

  * `cx1::Vector{Float64}`: See `cr1` for details.
  * `cnxa1::Vector{Union{Missing, Float64}}`: Winding connection angle in degrees; used when `cod1` is 5. There are no restrictions on the value specified for `cnxa1`; if it is outside of the range from -90.0 to +90.0, `cnxa1` is normalized to within this range. `cnxa1` = 0.0 by default.

  * `windv2::Vector{Float64}`: When `cw` is 1, `windv2` is the winding two off-nominal turns ratio in pu of winding two bus base voltage, and `windv2` = 1.0 by default. When `cw` is 2, `windv2` is the actual winding two voltage in kV, and `windv2` is equal to the base voltage of bus `j` by default.

  * `nomv2::Vector{Float64}`: The nominal (rated) winding two voltage in kV, or zero to indicate that nominal winding two voltage is to be taken as the base voltage of bus `j`. `nomv2` is present for information purposes only; it is not used in any of the calculations for modeling the transformer. `nomv2` = 0.0 by default.

  * `ang2::Vector{Union{Missing, Float64}}`: The winding two phase shift angle in degrees. `ang2` is positive for a positive phase shift from the winding two side to the "T" (or star) point bus. `ang2` must be greater than -180.0 and less than or equal to +180.0. `ang2` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `rata2::Vector{Union{Missing, Float64}}`: The second winding’s first rating entered in MVA (not current expressed in MVA). *Ignored for a two-winding transformer.*

  * `ratb2::Vector{Union{Missing, Float64}}`: The second winding’s second rating entered in MVA (not current expressed in MVA). *Ignored for a two-winding transformer.*

  * `ratc2::Vector{Union{Missing, Float64}}`: The second winding’s third rating entered in MVA (not current expressed in MVA). *Ignored for a two-winding transformer.*

  * `cod2::Vector{Union{Missing, Int8}}`: The transformer control mode for automatic adjustments of the winding two tap or phase shift angle during power flow solutions:

      * 0 for no control (fixed tap and phase shift);
      * ±1 for voltage control;
      * ±2 for reactive power flow control;
      * ±3 for active power flow control.

    If the control mode is entered as a positive number, automatic adjustment of this transformer winding is enabled when the corresponding adjustment is activated during power flow solutions; a negative control mode suppresses the automatic adjustment of this transformer winding. `cod2` = 0 by default. *Ignored for a two-winding transformer.*

  * `cont2::Vector{Union{Missing, Int32}}`: The bus number, or extended bus name enclosed in single quotes, of the bus whose voltage is to be controlled by the transformer turns ratio adjustment option of the power flow solution activities when `cod2` is 1. `cont2` should be nonzero only for voltage controlling transformer windings.

    `cont2` may specify a bus other than `i`, `j`, or `k`; in this case, the sign of `cont2` defines the location of the controlled bus relative to the transformer winding. If `cont2` is entered as a positive number, or a quoted extended bus name, the ratio is adjusted as if bus `cont2` is on the winding one or winding three side of the transformer; if `cont2` is entered as a negative number, or a quoted extended bus name with a minus sign preceding the first character, the ratio is adjusted as if bus `|cont2|` is on the winding two side of the transformer. `cont2` = 0 by default. *Ignored for a two-winding transformer.*

  * `rma2::Vector{Union{Missing, Float64}}`: `rma2` is the upper limit (and `rmi2` the lower limit) of either:

      * Off-nominal turns ratio in pu of winding two bus base voltage when `|cod2|` is 1 or 2 and `cw` is 1;   `rma2` = 1.1 and `rmi2` = 0.9 by default.
      * Actual winding one voltage in kV when `|cod2|` is 1 or 2 and `cw` is 2. No default is allowed.
      * Phase shift angle in degrees when `|cod2|` is 3. No default is allowed.
      * Not used when `|cod2|` is 0;

    `rma2` = 1.1 and `rmi2` = 0.9 by default. *Ignored for a two-winding transformer.*

  * `rmi2::Vector{Union{Missing, Float64}}`: The lower limit to `rma2`'s upper limit. See `rma2` for details. *Ignored for a two-winding transformer.*

  * `vma2::Vector{Union{Missing, Float64}}`: `vma2` is the upper limit (and `vmi2` the lower limit) of either:

      * Voltage at the controlled bus (bus `|cont2|`) in pu when `|cod2|` is 1.   `vma2` = 1.1 and `vmi2` = 0.9 by default.
      * Reactive power flow into the transformer at the winding two bus end in Mvar when `|cod2|` is 2.   No default is allowed.
      * Active power flow into the transformer at the winding two bus end in MW when `|cod2|` is 3.   No default is allowed.
      * Not used when `|cod2|` is 0; `vma2` = 1.1 and `vmi2` = 0.9 by default.

    *Ignored for a two-winding transformer.*

  * `vmi2::Vector{Union{Missing, Float64}}`: The lower limit to `vma1`'s upper limit. See `vma1` for details. *Ignored for a two-winding transformer.*

  * `ntp2::Vector{Union{Missing, Int16}}`: The number of tap positions available; used when `cod2` is 1 or 2. `ntp2` must be between 2 and 9999. `ntp2` = 33 by default. *Ignored for a two-winding transformer.*

  * `tab2::Vector{Union{Missing, Int64}}`: The number of a transformer impedance correction record if this transformer winding’s impedance is to be a function of either off-nominal turns ratio or phase shift angle, or 0 if no transformer impedance correction is to be applied to this transformer winding. See [`ImpedanceCorrections`](@ref). `tab2` = 0 by default. *Ignored for a two-winding transformer.*

  * `cr2::Vector{Union{Missing, Float64}}`: The load drop compensation impedance for voltage controlling transformers entered in pu on system base quantities; used when `cod2` is 1. `cr2` + j`cx2` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `cx2::Vector{Union{Missing, Float64}}`: See `cr2` for details. *Ignored for a two-winding transformer.*

  * `cnxa2::Vector{Union{Missing, Float64}}`: Winding connection angle in degrees; used when `cod2` is 5. There are no restrictions on the value specified for `cnxa2`; if it is outside of the range from -90.0 to +90.0, `cnxa2` is normalized to within this range. `cnxa2` = 0.0 by default.

  * `windv3::Vector{Union{Missing, Float64}}`: When `cw` is 1, `windv3` is the winding three off-nominal turns ratio in pu of winding three bus base voltage, and windv3 = 1.0 by default. When `cw` is 2, `windv3` is the actual winding three voltage in kV, and `windv3` is equal to the base voltage of bus `k` by default. *Ignored for a two-winding transformer.*

  * `nomv3::Vector{Union{Missing, Float64}}`: The nominal (rated) winding three voltage in kV, or zero to indicate that nominal winding two voltage is to be taken as the base voltage of bus `j`. `nomv3` is present for information purposes only; it is not used in any of the calculations for modeling the transformer. `nomv3` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `ang3::Vector{Union{Missing, Float64}}`: The winding three phase shift angle in degrees. `ang3` is positive for a positive phase shift from the winding two side to the "T" (or star) point bus. `ang3` must be greater than -180.0 and less than or equal to +180.0. `ang3` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `rata3::Vector{Union{Missing, Float64}}`: The third winding’s first rating entered in MVA (not current expressed in MVA). *Ignored for a two-winding transformer.*

  * `ratb3::Vector{Union{Missing, Float64}}`: The third winding’s second rating entered in MVA (not current expressed in MVA). *Ignored for a two-winding transformer.*

  * `ratc3::Vector{Union{Missing, Float64}}`: The third winding’s third rating entered in MVA (not current expressed in MVA). *Ignored for a two-winding transformer.*

  * `cod3::Vector{Union{Missing, Int8}}`: The transformer control mode for automatic adjustments of the winding three tap or phase shift angle during power flow solutions:

      * 0 for no control (fixed tap and phase shift);
      * ±1 for voltage control;
      * ±2 for reactive power flow control;
      * ±3 for active power flow control.

    If the control mode is entered as a positive number, automatic adjustment of this transformer winding is enabled when the corresponding adjustment is activated during power flow solutions; a negative control mode suppresses the automatic adjustment of this transformer winding. `cod3` = 0 by default. *Ignored for a two-winding transformer.*

  * `cont3::Vector{Union{Missing, Int32}}`: The bus number, or extended bus name enclosed in single quotes, of the bus whose voltage is to be controlled by the transformer turns ratio adjustment option of the power flow solution activities when `cod3` is 1. `cont3` should be nonzero only for voltage controlling transformer windings.

    `cont3` may specify a bus other than `i`, `j`, or `k`; in this case, the sign of `cont3` defines the location of the controlled bus relative to the transformer winding. If `cont3` is entered as a positive number, or a quoted extended bus name, the ratio is adjusted as if bus `cont3` is on the winding one or winding two side of the transformer; if `cont3` is entered as a negative number, or a quoted extended bus name with a minus sign preceding the first character, the ratio is adjusted as if bus `|cont3|` is on the winding three side of the transformer. `cont3` = 0 by default. *Ignored for a two-winding transformer.*

  * `rma3::Vector{Union{Missing, Float64}}`: `rma3` is the upper limit (and `rmi3` the lower limit) of either:

      * Off-nominal turns ratio in pu of winding three bus base voltage when `|cod3|` is 1 or 2 and `cw` is 1; `rma3` = 1.1 and `rmi3` = 0.9 by default.
      * Actual winding one voltage in kV when `|cod3|` is 1 or 2 and `cw` is 2. No default is allowed.
      * Phase shift angle in degrees when `|cod3|` is 3. No default is allowed.
      * Not used when `|cod3|` is 0;

    `rma3` = 1.1 and `rmi3` = 0.9 by default. *Ignored for a two-winding transformer.*

  * `rmi3::Vector{Union{Missing, Float64}}`: The lower limit to `rma3`'s upper limit. See `rma3` for details. *Ignored for a two-winding transformer.*

  * `vma3::Vector{Union{Missing, Float64}}`: `vma3` is the upper limit (and `vmi3` the lower limit) of either:

      * Voltage at the controlled bus (bus `|cont3|`) in pu when `|cod3|` is 1. `vma3` = 1.1 and `vmi3` = 0.9 by default.
      * Reactive power flow into the transformer at the winding three bus end in Mvar when `|cod3|` is 2. No default is allowed.
      * Active power flow into the transformer at the winding two bus end in MW when `|cod3|` is 3. No default is allowed.
      * Not used when `|cod3|` is 0; `vma3` = 1.1 and `vmi3` = 0.9 by default.

    *Ignored for a two-winding transformer.*

  * `vmi3::Vector{Union{Missing, Float64}}`: The lower limit to `vma3`'s upper limit. See `vma3` for details. *Ignored for a two-winding transformer.*

  * `ntp3::Vector{Union{Missing, Int16}}`: The number of tap positions available; used when `cod3` is 1 or 2. `ntp3` must be between 2 and 9999. `ntp3` = 33 by default. *Ignored for a two-winding transformer.*

  * `tab3::Vector{Union{Missing, Int64}}`: The number of a transformer impedance correction record if this transformer winding’s impedance is to be a function of either off-nominal turns ratio or phase shift angle, or 0 if no transformer impedance correction is to be applied to this transformer winding. See [`ImpedanceCorrections`](@ref). `tab3` = 0 by default. *Ignored for a two-winding transformer.*

  * `cr3::Vector{Union{Missing, Float64}}`: The load drop compensation impedance for voltage controlling transformers entered in pu on system base quantities; used when `cod3` is 1. `cr3` + j`cx3` = 0.0 by default. *Ignored for a two-winding transformer.*

  * `cx3::Vector{Union{Missing, Float64}}`: See `cr3` for details. *Ignored for a two-winding transformer.*

  * `cnxa3::Vector{Union{Missing, Float64}}`: Winding connection angle in degrees; used when `cod3` is 5. There are no restrictions on the value specified for `cnxa3`; if it is outside of the range from -90.0 to +90.0, `cnxa3` is normalized to within this range. `cnxa3` = 0.0 by default.


In [37]:
case.transformers

Transformers with 4 records, 43 columns:
────────────────────────────────────────────────────────────────────────────────
 i  j  k  ckt  cw  cz  cm  mag1  mag2  nmetr  name  stat  o1   f1  o2   f2  o3 ⋯
────────────────────────────────────────────────────────────────────────────────
 4  7  0    1   1   1   1   0.0   0.0      2        true   1  1.0   0  1.0   0 ⋯
 4  9  0    1   1   1   1   0.0   0.0      2        true   1  1.0   0  1.0   0 ⋯
 6  5  0    1   1   1   1   0.0   0.0      1        true   1  1.0   0  1.0   0 ⋯
 8  7  0    1   1   1   1   0.0   0.0      1        true   1  1.0   0  1.0   0 ⋯
────────────────────────────────────────────────────────────────────────────────
                                                              26 columns omitted